In [189]:
from scipy.io import savemat, loadmat
import matplotlib.pyplot as plt

# RANDOM SEED
# Fix the random seed so the same dataset is generated every time the script is executed
np.random.seed(42)

# FIXED PARAMETERS
# Membrane capacitance
C = 1.0
# Reversal potentials (mV)
VNa = 60.0
VK = -90.0
VL = -80.0
# Leak conductance
gL = 8.0
# Parameters of the steady-state activation functions
V12m = -20.0
km = 15.0
V12n = -25.0
kn = 5.0
# Reference sodium conductance
gNa0 = 20.0

# ACTIVATION FUNCTIONS
# Steady-state sodium activation
def m_inf(V):
    return 1.0 / (1.0 + np.exp((V12m - V) / km))
# Steady-state potassium activation
def n_inf(V):
    return 1.0 / (1.0 + np.exp((V12n - V) / kn))

# ODE SYSTEM
# Compute the derivatives of the four state variables
def rhs(state, gK0, tau_n, eps, Iapp):
    # Unpack the current state
    V, n, gNa, gK = state
    # Membrane voltage equation
    dV = (
        Iapp
        - gL * (V - VL)
        - gNa * m_inf(V) * (V - VNa)
        - gK * n * (V - VK)
    ) / C
    # Potassium activation relaxes towards its steady state
    dn = (n_inf(V) - n) / tau_n
    # Slow conductance dynamics
    dgNa = eps * (gK0 - gK)
    dgK = eps * (gNa - gNa0)
    return np.array([dV, dn, dgNa, dgK])

# RK4
# Perform one integration step using the classical fourth-order Runge-Kutta method for improved accuracy
def rk4_step(state, dt, gK0, tau_n, eps, Iapp):

    k1 = rhs(state, gK0, tau_n, eps, Iapp)
    k2 = rhs(state + 0.5 * dt * k1, gK0, tau_n, eps, Iapp)
    k3 = rhs(state + 0.5 * dt * k2, gK0, tau_n, eps, Iapp)
    k4 = rhs(state + dt * k3, gK0, tau_n, eps, Iapp)

    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

# SIMULATION
# Simulate the neuron dynamics for a given parameter set
def simulate(gK0, tau_n, eps, Iapp, T=1000, dt=0.05):
    # Number of integration steps
    N = int(T / dt)
    # Preallocate memory for efficiency
    time = np.zeros(N + 1, dtype=np.float32)
    V_trace = np.zeros(N + 1, dtype=np.float32)
    n_trace = np.zeros(N + 1, dtype=np.float32)
    gNa_trace = np.zeros(N + 1, dtype=np.float32)
    gK_trace = np.zeros(N + 1, dtype=np.float32)
    # Initial conditions gNa starts slightly above its equilibrium value to help the system leave the initial transient.
    state = np.array([
        -60.0,
        0.0,
        gNa0 + 1,
        gK0
    ])

    # Store the initial state
    V_trace[0] = state[0]
    n_trace[0] = state[1]
    gNa_trace[0] = state[2]
    gK_trace[0] = state[3]
    # Integrate the ODE system over time
    for i in range(N):
        state = rk4_step(
            state,
            dt,
            gK0,
            tau_n,
            eps,
            Iapp
        )

        # Save the updated state
        V_trace[i + 1] = state[0]
        n_trace[i + 1] = state[1]
        gNa_trace[i + 1] = state[2]
        gK_trace[i + 1] = state[3]
        time[i + 1] = time[i] + dt
    return time, V_trace, n_trace, gNa_trace, gK_trace

# DATASET PARAMETERS
# Number of simulations generated for each bursting class
Nsim_class = 500
# Default simulation parameters
tau_n = 2
eps = 0.0005
Iapp = 1.7
# Each gK0 value defines a different bursting regime
gK0_values = [
    (11.0, 0),
    (9.75, 1),
    (9, 2)
]

# GENERATE DATASET
# Lists used to store the complete dataset
X_all = []
Y_all = []
labels = []
# Generate samples for each bursting class
for gK0_center, label in gK0_values:
    print(f"Generating class {label}")
    for _ in range(Nsim_class):
        # Slightly perturb gK0 so that samples within the same class are not exactly identical
        gK0 = gK0_center + np.random.uniform(-0.10, 0.10)
        # Additional parameter variations
        # These are stored as inputs for the operator
        Iapp_sample = np.random.uniform(1.8, 2.2)
        tau_sample = np.random.uniform(0.15, 0.19)
        eps_sample = np.random.uniform(0.0013, 0.0019)
        # Simulate one trajectory
        t, V, n, gNa, gK = simulate(
            gK0=gK0,
            tau_n=tau_n,
            eps=eps,
            Iapp=Iapp
        )
        # Store the input parameters
        X_all.append([
            gK0,
            Iapp_sample,
            tau_sample,
            eps_sample
        ])
        # Store the four simulated state variables
        Y_all.append(
            np.stack(
                [V, n, gNa, gK],
                axis=-1
            )
        )
        # Store the class label
        labels.append(label)

# CONVERT TO ARRAYS
# Convert lists into NumPy arrays for efficient storage and compatibility with DeepONet
X = np.array(X_all, dtype=np.float32)
U = np.array(Y_all, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

# DATASET (WITHOUT NORMALIZATION)
print("Dataset generated without normalization.")
print("Voltage range:")
print("Min:", U.min())
print("Max:", U.max())

# SAVE MATLAB FILE
# Save the dataset in MATLAB format so it can be loaded directly by the DeepONet implementation
savemat(
    "dataset_raw.mat",
    {
        "X": X,
        "time": t,
        "V": U,
        "label": labels,
    }
)

# FINAL INFORMATION
print("\nDataset generated successfully\n")
print("X :", X.shape)
print("V :", U.shape)
print("Labels :", labels.shape)

# LOAD AND VISUALIZE ONE SIMULATION
# Load the generated dataset
data = loadmat("dataset_raw.mat")
X = data["X"]
V = data["V"]
labels = data["label"].flatten()
t = data["time"].flatten()

# Select the simulation to visualize
idx = 0      # Change this index to inspect another sample
print("Selected simulation")
print("Index:", idx)
print("Class:", labels[idx])
print("Input parameters:", X[idx])

# Plot the membrane voltage trace
# V contains the four state variables (V, n, gNa and gK), so the first column corresponds to the membrane voltage
plt.figure(figsize=(14, 4))
plt.plot(t, V[idx][:, 0])
plt.xlabel("Time")
plt.ylabel("Voltage (mV)")
plt.title(f"Simulation {idx}   Class = {labels[idx]}")
plt.grid(True)
plt.show()

Generating class 0


KeyboardInterrupt: 

In [188]:
import numpy as np
from scipy.io import loadmat, savemat
from sklearn.model_selection import train_test_split

# FILES
# Input dataset generated by the simulation script
INPUT_FILE = "dataset_raw.mat"
# Output files for training and testing
TRAIN_FILE = "train_bursts.mat"
TEST_FILE = "test_bursts.mat"
# Fraction of samples reserved for testing
TEST_SIZE = 0.20
# Fix the random state to obtain the same split every run
RANDOM_STATE = 42

# LOAD DATASET
print("Loading dataset...")
# Load the complete dataset
data = loadmat(INPUT_FILE)
# Input parameters
X = data["X"]
# Simulated trajectories
V = data["V"]
# Shared time vector
time = data["time"].squeeze()
# Class labels
labels = data["label"].squeeze()
# Display dataset dimensions
print("X:", X.shape)
print("V:", V.shape)
print("Labels:", labels.shape)
print("Time:", time.shape)

# TRAIN / TEST SPLIT
# Split the dataset into training and testing subsets.
# Stratification preserves the class distribution in both sets.
X_train, X_test, \
V_train, V_test, \
y_train, y_test = train_test_split(
    X,
    V,
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels
)

# NORMALIZATION
# Compute normalization statistics using only the
# training set to avoid information leakage.
mean = np.mean(V_train)
std = np.std(V_train)

# Apply the same normalization to both datasets
V_train = (V_train - mean) / std
V_test = (V_test - mean) / std

# SAVE TRAIN
# Save the normalized training dataset together with
# the normalization statistics.
savemat(
    TRAIN_FILE,
    {
        "X": X_train,
        "V": V_train,
        "time": time,
        "label": y_train,
        "mean": np.array([mean], dtype=np.float32),
        "std": np.array([std], dtype=np.float32),
    }
)

# SAVE TEST
# Save the normalized testing dataset using the same
# normalization parameters as the training set.
savemat(
    TEST_FILE,
    {
        "X": X_test,
        "V": V_test,
        "time": time,
        "label": y_test,
        "mean": np.array([mean], dtype=np.float32),
        "std": np.array([std], dtype=np.float32),
    }
)


# SUMMARY
print("\nDataset successfully split.\n")
print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])

# Verify that the class distribution is preserved
print("\nTraining labels")
print(np.bincount(y_train))

print("\nTesting labels")
print(np.bincount(y_test))

# Display the normalization statistics
print("\nMean :", mean)
print("Std  :", std)

Loading dataset...
X: (30, 4)
V: (30, 20001, 4)
Labels: (30,)
Time: (20001,)

Dataset successfully split.

Training samples : 24
Testing samples  : 6

Training labels
[8 8 8]

Testing labels
[2 2 2]

Mean : -6.4015393
Std  : 32.61439
